# Introduction to GPU Programming with Triton

This notebook is an **introduction to Triton**, a Python-like language for writing high-performance GPU kernels, and how it relates to **CUDA programming concepts** like threads and blocks.  

---

## CUDA Basics

When programming GPUs with **CUDA**, there are a few key concepts to understand:

### Threads
- The smallest unit of execution on a GPU.
- Each thread computes a single part of a problem, like one element of a matrix.
- Think of a thread as a “worker” that performs a tiny task.

### Blocks
- Threads are grouped into **blocks**.
- Threads in a block can share **fast shared memory** and synchronize with each other.
- Each block computes a portion of the output, like a small tile of a matrix.

### Grids
- Blocks are organized into a **grid** to cover the entire problem.
- Together, all blocks compute the full output.


## What is a CUDA Thread?

In CUDA programming:

- A **thread** is the **smallest unit of execution on the GPU**.
- Each thread executes the **same kernel code**, but usually operates on **different data**.

**Thread indexing:**

- `threadIdx` — the thread’s index **within its block**.
- `blockIdx` — the block’s index **within the grid**.
- `blockDim` — the number of threads **per block**.
- To get the **global index** of a thread (across all blocks):


In [1]:
%time
import numpy as np
from numba import cuda


@cuda.jit
def vector_add_kernel(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    # Get the global thread index
    idx = cuda.grid(1)

    # Bounds check
    if idx < C.size:
        C[idx] = A[idx] + B[idx]


# Initialize vectors
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)  # decreasing values
C = np.zeros(n, dtype=np.float32)

print("Vector A:", A)
print("Vector B:", B)

# Configure threads and blocks
threads_per_block = 8
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel
vector_add_kernel[blocks_per_grid, threads_per_block](A, B, C)

print("Result Vector C:", C)

CPU times: user 2 µs, sys: 0 ns, total: 2 µs
Wall time: 5.48 µs
Vector A: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
Vector B: [16. 15. 14. 13. 12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Result Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


## What is a CUDA Block?

In CUDA programming:

- A **block** is a group of threads that execute together on a single **Streaming Multiprocessor (SM)** on the GPU.
- Threads in a block can:
  - Share **fast shared memory**.
  - Synchronize with each other using `cuda.syncthreads()`.
  
- Blocks are organized into a **grid** to cover larger datasets.

**Thread hierarchy:**

1. **Thread:** smallest unit of execution.
2. **Block:** a group of threads that can communicate via shared memory.
3. **Grid:** a group of blocks.

**Thread indexing within a block:**

- `threadIdx.x` — thread’s index **inside the block**.
- `blockIdx.x` — block’s index **inside the grid**.
- `blockDim.x` — number of threads **per block**.

**Global thread index formula:**


```global_idx = threadIdx.x + blockIdx.x * blockDim.x```



Each block is independent; threads in different blocks **cannot directly share shared memory**.


In [2]:
%time
import numpy as np
from numba import cuda


# CUDA kernel for vector addition
@cuda.jit
def vector_add_kernel(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    # Thread index inside block
    tx = cuda.threadIdx.x

    # Block index inside grid
    bx = cuda.blockIdx.x

    # Number of threads per block
    bdim = cuda.blockDim.x

    # Compute global thread index
    idx = tx + bx * bdim

    # Bounds check
    if idx < C.size:
        C[idx] = A[idx] + B[idx]


# Initialize vectors
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)
C = np.zeros(n, dtype=np.float32)

print("Vector A:", A)
print("Vector B:", B)

# Configure threads per block and number of blocks
threads_per_block = 4
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel
vector_add_kernel[blocks_per_grid, threads_per_block](A, B, C)

print("Result Vector C:", C)

CPU times: user 3 µs, sys: 1 µs, total: 4 µs
Wall time: 6.68 µs
Vector A: [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]
Vector B: [16. 15. 14. 13. 12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]
Result Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


## Why Use Blocks in CUDA?

Using blocks is essential for **performance and scalability**:

1. **Shared Memory:**
   - Threads within the same block can share **fast shared memory**.
   - This avoids repeatedly reading the same data from **slower global memory**.

2. **Thread Cooperation:**
   - Threads in a block can **synchronize** using `cuda.syncthreads()`.
   - Useful for reductions, scans, or any operation where threads depend on each other.

3. **Tiling / Block-wise Computation:**
   - Large datasets can be **split into blocks (tiles)**.
   - Each block loads a chunk of data into shared memory, computes results locally, and writes back.
   - This **reduces global memory accesses** and improves performance.

We’ll demonstrate this with **tiled vector addition**, where each block loads its portion of two vectors into shared memory, adds them, and writes back the results.

## Tiled Vector Addition

Imagine we have:

- Vector A = [0, 1, 2, 3, 4, 5, 6, 7]  
- Vector B = [8, 7, 6, 5, 4, 3, 2, 1]  
- Threads per block = 4 → 2 blocks

---

### Step 1: Divide into Blocks (Tiles)

| Block | Thread IDs (tx) | Global Index (idx) |
|-------|----------------|------------------|
| 0     | 0,1,2,3       | 0,1,2,3         |
| 1     | 0,1,2,3       | 4,5,6,7         |

---

### Step 2: Load Tile into Shared Memory
```
Block 0 Shared Memory (sA, sB):
sA = [0,1,2,3]
sB = [8,7,6,5]

Block 1 Shared Memory (sA, sB):
sA = [4,5,6,7]
sB = [4,3,2,1]
```


- Each thread loads **one element** into shared memory.
- Threads in the block synchronize with `cuda.syncthreads()`.

---

### Step 3: Compute in Shared Memory
```
Block 0:
Thread 0: sA[0] + sB[0] = 0 + 8 = 8
Thread 1: sA[1] + sB[1] = 1 + 7 = 8
Thread 2: sA[2] + sB[2] = 2 + 6 = 8
Thread 3: sA[3] + sB[3] = 3 + 5 = 8

Block 1:
Thread 0: sA[0] + sB[0] = 4 + 4 = 8
Thread 1: sA[1] + sB[1] = 5 + 3 = 8
Thread 2: sA[2] + sB[2] = 6 + 2 = 8
Thread 3: sA[3] + sB[3] = 7 + 1 = 8
```

---

### Step 4: Write Back to Global Memory

- Each thread writes its computed value from **shared memory** to **global memory**.
- Multiple blocks work independently in parallel.


In [3]:
import numpy as np
from numba import cuda

MAX_SIZE = 32  # choose a safe upper bound for threads per block


# CUDA kernel: Tiled vector addition using "static" shared memory with flexible size
@cuda.jit
def vector_add_tiled(A: np.ndarray, B: np.ndarray, C: np.ndarray, sA_size: int, sB_size: int):
    # Declare max possible shared memory size
    sA = cuda.shared.array(MAX_SIZE, dtype=cuda.float32)
    sB = cuda.shared.array(MAX_SIZE, dtype=cuda.float32)

    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdim = cuda.blockDim.x
    idx = tx + bx * bdim

    # Only use the portion of shared memory according to passed sizes
    if idx < C.size and tx < sA_size:
        sA[tx] = A[idx]
    if idx < C.size and tx < sB_size:
        sB[tx] = B[idx]
    cuda.syncthreads()

    if idx < C.size and tx < sA_size and tx < sB_size:
        sA[tx] = sA[tx] + sB[tx]
    cuda.syncthreads()

    if idx < C.size and tx < sA_size:
        C[idx] = sA[tx]


# Data
n = 16
A = np.arange(n, dtype=np.float32)
B = np.arange(n, 0, -1, dtype=np.float32)
C = np.zeros(n, dtype=np.float32)

threads_per_block = 4
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

# Launch kernel with sizes for sA and sB
vector_add_tiled[blocks_per_grid, threads_per_block](A, B, C, threads_per_block, threads_per_block)

print("Vector C:", C)

Vector C: [16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16. 16.]


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


# Triton Basics

> **Block-size requirement:** When using `tl.arange(0, BLOCK_SIZE)` in Triton, the block size must be a **power of two** (`2, 4, 8, 16, 32, ...`), not merely an even number. For example, `8` and `16` are valid, but `3`, `5`, `6`, and `10` are not. Input dimensions do **not** have to be powers of two: use a power-of-two block size and mask indices outside the input bounds.

Triton simplifies GPU programming by **removing much of the low-level thread/block management**.  

## Program
- In Triton, a **program** is like a “super-thread” that can operate on **a whole tile of data**.
- A program handles multiple elements at once using **vectorized operations**.
- Triton automatically maps these programs onto GPU threads and warps.

## Threads vs Programs
| Concept | CUDA | Triton |
|---------|------|--------|
| Smallest unit | Thread (1 element) | Program (1 tile of elements) |
| Shared memory | Threads in a block | Program uses fast memory implicitly |
| Mapping | ThreadIdx / BlockIdx | tl.program_id() |

## Tiled Vector Addition Using Triton

In Triton:

- A **kernel** is written as a Python function decorated with `@triton.jit`.
- Threads are organized in **blocks**, called **program IDs**.
- We can load **tiles of data into shared memory (using Triton’s `tl.load`)** for faster computation.
- Each thread (or “program”) handles a portion of the tile.

We will implement **vector addition C = A + B** using **tiled blocks**.

In [4]:
import torch
import triton
import triton.language as tl
%time

# Triton kernel: vector addition with tiling
@triton.jit
def vector_add_tiled_kernel(
        A_ptr: torch.Tensor,
        B_ptr: torch.Tensor,
        C_ptr: torch.Tensor,
        N: tl.constexpr,
        BLOCK_SIZE: tl.constexpr
):
    # program ID = block index
    pid = tl.program_id(0)

    # compute start index of this block
    start = pid * BLOCK_SIZE

    # compute offsets for threads in the block
    offsets = start + tl.arange(0, BLOCK_SIZE)

    # Mask for threads that go out of bounds
    mask = offsets < N

    # Load from global memory
    a = tl.load(A_ptr + offsets, mask=mask)
    b = tl.load(B_ptr + offsets, mask=mask)

    # Compute addition
    c = a + b

    # Write back to global memory
    tl.store(C_ptr + offsets, c, mask=mask)


# Example vectors
N = 16
BLOCK_SIZE = 4
A = torch.arange(N, dtype=torch.float32, device='cuda')
B = torch.arange(N, 0, -1, dtype=torch.float32, device='cuda')
C = torch.zeros(N, dtype=torch.float32, device='cuda')

print("Vector A:", A)
print("Vector B:", B)

# Launch kernel
grid = (triton.cdiv(N, BLOCK_SIZE),)
vector_add_tiled_kernel[grid](A, B, C, N, BLOCK_SIZE=BLOCK_SIZE)

print("Result Vector C:", C)

CPU times: user 1 µs, sys: 0 ns, total: 1 µs
Wall time: 4.05 µs
Vector A: tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15.], device='cuda:0')
Vector B: tensor([16., 15., 14., 13., 12., 11., 10.,  9.,  8.,  7.,  6.,  5.,  4.,  3.,
         2.,  1.], device='cuda:0')
Result Vector C: tensor([16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16.,
        16., 16.], device='cuda:0')


## Why Tiled Vector Addition in Triton May Not Give Performance Boost

1. **Vector addition is memory-bound**:  
   - Each element is **read once from A and B, written once to C**.  
   - GPU arithmetic is cheap; most of the time is spent moving data.  

2. **Tiling only helps when data is reused**:  
   - In vector addition, each element is used **exactly once**.  
   - Loading tiles into shared memory (registers) **does not reduce global memory traffic**, because we still read each element once.  

3. **Overhead of tiling**:  
   - Extra instructions to compute offsets and masks may **slightly increase runtime** for small vectors.  

## Triton Matrix Multiplication Using Tiling


We are computing **C = A @ B**, where:

- A: (M x K)  
- B: (K x N)  
- C: (M x N)  

We use **tiled blocks** to exploit GPU parallelism and fast memory.

---

## 1. Block-level parallelism

- Each block computes a **tile of C** with size `BLOCK_M x BLOCK_N`.
- **Program IDs** identify each block:

```
pid_m = tl.program_id(0) # block row index
pid_n = tl.program_id(1) # block column index
```

- Block start indices:

```
m_start = pid_m * BLOCK_M
n_start = pid_n * BLOCK_N
```

- Example:  

```
BLOCK_M = 8, BLOCK_N = 8
pid_m=0, pid_n=0 -> computes C[0:8, 0:8]
pid_m=1, pid_n=0 -> computes C[8:16, 0:8]

```

---

## 2. Tiles and shared registers

- **accumulator**: each block creates a local `BLOCK_M x BLOCK_N` matrix in **registers**:


```acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)```


- This holds the **partial sum** of the block’s tile of C.
- Each element of the tile is computed by **multiple multiply-adds** over the K dimension.

---

## 3. Looping over K dimension in tiles

- The K dimension is usually too large to load fully into fast memory.
- Loop over K in **BLOCK_K chunks**:


```
for k_start in range(0, K, BLOCK_K):
    load A_tile: shape (BLOCK_M x BLOCK_K)
    load B_tile: shape (BLOCK_K x BLOCK_N)
    acc += A_tile @ B_tile
```

- Each tile is loaded from **global memory** into registers.  
- **Multiply-accumulate (tl.dot)** is performed in fast memory, **reducing global memory accesses**.

---

## 4. Offsets and masking

- Compute **global indices** for each tile:


```
offs_m = m_start + tl.arange(0, BLOCK_M)
offs_n = n_start + tl.arange(0, BLOCK_N)
offs_k = k_start + tl.arange(0, BLOCK_K)
```

- `tl.load` uses a **mask** to avoid out-of-bounds reads:


```mask=(offs_m[:, None] < M) & (offs_k[None, :] < K)```


- This ensures safety for the **last blocks** if M, N, or K are not multiples of tile size.

---

## 5. Multiply-accumulate (tl.dot)

- Compute:


```acc += tl.dot(A_tile, B_tile)```


- Each element in the tile of C is the **sum of products** along the K dimension.  
- After looping over all K tiles, `acc` contains the **final values** for this block’s C tile.

---

## 6. Write back to global memory

- After all K tiles are processed:


```tl.store(C_ptr + offs_m[:, None] * N + offs_n[None, :], acc, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))```


- Each block writes its tile of C to the correct location in global memory.  
- Multiple blocks write **independently in parallel**, covering the entire matrix.

---

## 7. Summary of Benefits

- **Tiling reduces global memory traffic**: each A and B element is loaded **once per tile** but used many times.  
- **Registers hold temporary results**, avoiding slow global memory for intermediate computations.  
- **Blocks operate independently** → full GPU parallelism.  
- **Masking ensures safety** for non-multiple dimensions.  

### Animation: tiled matrix multiplication

At each step along the **K dimension**, a program multiplies a tile from **A** by a tile from **B**, then adds the partial product to its output accumulator (`Result += Temp`). It repeats this until K is covered; other programs compute different output tiles in parallel.

![Animated example of accumulating matrix products over K tiles](assets/tmm.gif)

In [5]:
import torch
import triton
import triton.language as tl


@triton.jit
def matmul_kernel(
        A_ptr: torch.Tensor,  # tensor A
        B_ptr: torch.Tensor,  # tensor B
        C_ptr: torch.Tensor,  # tensor C
        M: tl.constexpr,  # number of rows in A/C
        N: tl.constexpr,  # number of columns in B/C
        K: tl.constexpr,  # number of columns in A / rows in B
        BLOCK_M: tl.constexpr,  # tile height
        BLOCK_N: tl.constexpr,  # tile width
        BLOCK_K: tl.constexpr  # tile depth along K
):
    # Which tile of C this program computes
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    # Accumulator for the BLOCK_M x BLOCK_N tile of C (kept in registers)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # Loop over K in BLOCK_K chunks
    for k_start in range(0, K, BLOCK_K):
        offs_k = k_start + tl.arange(0, BLOCK_K)

        # A is row-major (M x K): element (i, k) is at i * K + k
        a_ptrs = A_ptr + offs_m[:, None] * K + offs_k[None, :]
        a_mask = (offs_m[:, None] < M) & (offs_k[None, :] < K)
        a_tile = tl.load(a_ptrs, mask=a_mask, other=0.0)

        # B is row-major (K x N): element (k, j) is at k * N + j
        b_ptrs = B_ptr + offs_k[:, None] * N + offs_n[None, :]
        b_mask = (offs_k[:, None] < K) & (offs_n[None, :] < N)
        b_tile = tl.load(b_ptrs, mask=b_mask, other=0.0)

        # Padded elements are zeros, so they don't affect the sum
        acc += tl.dot(a_tile, b_tile, input_precision="ieee")

    # Write the tile of C back to global memory
    c_ptrs = C_ptr + offs_m[:, None] * N + offs_n[None, :]
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, acc, mask=c_mask)


# Example usage
M, N, K = 16, 16, 16
BLOCK_M, BLOCK_N, BLOCK_K = 16, 16, 32  # power-of-two tiles; compatible with typical tl.dot constraints

A = torch.randn((M, K), dtype=torch.float32, device='cuda')
B = torch.randn((K, N), dtype=torch.float32, device='cuda')
C = torch.zeros((M, N), dtype=torch.float32, device='cuda')

grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
matmul_kernel[grid](A, B, C, M, N, K, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K)

print("Matrix C:\n", C)
print("Match torch?", torch.allclose(C, A @ B, atol=1e-4))

Matrix C:
 tensor([[-4.7704e+00,  2.7371e+00,  3.1811e+00,  2.2192e+00, -6.0120e-01,
          3.9928e-01,  2.7847e+00,  3.9183e+00, -1.9173e+00,  1.5555e+00,
         -2.5410e+00, -4.4668e-01,  7.1997e+00, -3.7893e+00,  2.8797e+00,
         -3.0367e+00],
        [ 1.6810e+00,  1.3642e+00, -1.2164e+00,  1.0453e+00, -4.6960e+00,
          4.2793e+00,  9.5009e-01, -4.4882e+00,  3.7618e+00, -2.9522e+00,
         -8.2632e-01, -1.9034e+00, -1.0880e+01,  9.5875e-01, -9.4934e-01,
          5.5311e-01],
        [ 1.2375e+00, -5.7065e-01, -2.2039e+00, -3.6147e+00, -1.6354e+00,
          1.4323e+00,  1.4143e+00, -1.2480e+00,  6.3336e+00,  2.3191e+00,
          8.4149e-01,  2.6815e+00,  1.4052e+00, -2.1475e+00, -2.9527e+00,
          1.6767e+00],
        [ 6.9898e-02, -4.3892e+00, -2.8812e+00, -1.5950e+00, -4.5442e+00,
         -1.5084e+00,  7.9407e+00, -3.6609e+00,  2.9934e+00,  2.9921e+00,
          6.6278e-01, -1.4107e+00, -3.8093e+00, -1.1109e+00, -2.2199e+00,
         -1.3333e+00],
        [

In [6]:
@triton.jit
def _seeded_dropout(
        x_ptr: torch.Tensor,
        output_ptr: torch.Tensor,
        n_elements: int,
        p: float,
        seed: int,
        BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    # случайные числа считаются из (seed, offset), маску хранить не нужно
    random = tl.rand(seed, offsets)
    keep = random > p
    output = tl.where(keep, x / (1 - p), 0.0)
    tl.store(output_ptr + offsets, output, mask=mask)


def seeded_dropout(x: torch.Tensor, p: float, seed: int) -> torch.Tensor:
    output = torch.empty_like(x)
    assert x.is_contiguous()
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)
    _seeded_dropout[grid](x, output, n_elements, p, seed, BLOCK_SIZE=1024)
    return output


x = torch.randn(size=(10,)).cuda()
# Compare this to the baseline - dropout mask is never instantiated!
output = seeded_dropout(x, p=0.5, seed=123)

print("Input: ", x)
print("Out: ", output)

Input:  tensor([-1.2408, -0.0165,  0.7412,  0.4324, -0.2638, -0.5535,  0.4438, -0.4821,
        -0.3869,  1.1874], device='cuda:0')
Out:  tensor([ 0.0000, -0.0330,  0.0000,  0.0000,  0.0000, -1.1070,  0.0000,  0.0000,
        -0.7738,  2.3749], device='cuda:0')


## `torch.compile` and TorchInductor

`torch.compile` speeds up ordinary PyTorch code **without manually writing GPU kernels**. **TorchDynamo** captures operations as a graph, and **TorchInductor** (`backend="inductor"`) optimizes it (e.g., fusing operations into Triton kernels on NVIDIA GPUs).

The first call includes compilation, so **benchmark after warm-up**.

| Mode | When to use it |
|---|---|
| `default` | Balanced compilation time and performance. |
| `reduce-overhead` | Uses **CUDA Graphs** to reduce launch overhead for repeated GPU work; may use extra memory. |
| `max-autotune` | Searches for faster kernel configurations; slower to compile and enables CUDA Graphs on GPU. |
| `max-autotune-no-cudagraphs` | Autotuning without CUDA Graphs. |

CUDA Graphs **record and replay GPU operations**, reducing repeated launch overhead. Not every workload supports them.


In [7]:
import torch

# PyTorch operations are fused/optimized automatically.
def example_op(x):
    return torch.relu(torch.sin(x) + 0.5 * x)

x_compile = torch.randn(65536, device="cuda")
compiled_op = torch.compile(example_op, backend="inductor")  # default mode

# First call triggers compilation.
torch.testing.assert_close(compiled_op(x_compile), example_op(x_compile))


In [8]:
import triton.testing

# Try: "default", "reduce-overhead", "max-autotune",
#      "max-autotune-no-cudagraphs"
mode = "reduce-overhead"
compiled_mode_op = torch.compile(example_op, backend="inductor", mode=mode)
compiled_mode_op(x_compile)  # compile/warm-up before measuring

print("PyTorch (ms):", triton.testing.do_bench(lambda: example_op(x_compile)))
print(f"Compiled ({mode}) (ms):", triton.testing.do_bench(lambda: compiled_mode_op(x_compile)))


PyTorch (ms): 0.024044952466196957
Compiled (reduce-overhead) (ms): 0.0388889113511843


# Homework

Implement the following operators using Triton. Complete the provided kernel stubs and verify their outputs against the PyTorch references.

**Important: Triton block sizes must be powers of two** when used with `tl.arange(0, BLOCK_SIZE)` (e.g., `2, 4, 8, 16, 32, 64, 128`), **not just divisible by 2**. In these exercises each kernel launch uses one program per row or column, so the configured block must cover the full row/column (`BLOCK_N >= N`, `BLOCK_M >= M`, or `BLOCK_F >= F`). Round up with `triton.next_power_of_2(dimension)` and use masking for padded elements (e.g., a dimension of `13` uses block size `16`).


# Task 1: Row-wise Matrix Sum

**Problem:**  

Implement a Triton kernel that computes **the sum of each row** of a matrix.  

- Input: `A` of shape `(M, N)`  
- Output: `row_sums` of shape `(M,)`  
- Hint:  
  - Use **tiles along the N dimension**  
  - Accumulate the sum in **registers**  
  - Handle partial tiles with **masking**

In [9]:
import torch
import triton
import triton.language as tl


@triton.jit
def row_sum_kernel(A_ptr: torch.Tensor,
                   row_sums_ptr: torch.Tensor,
                   M: int, N: int,
                   BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    acc = tl.zeros((BLOCK_N,), dtype=tl.float32)
    for start in range(0, N, BLOCK_N):
        cols = start + tl.arange(0, BLOCK_N)
        acc += tl.load(A_ptr + row * N + cols, mask=cols < N, other=0.0)
    tl.store(row_sums_ptr + row, tl.sum(acc, axis=0))


def run_kernel(A, BLOCK_N):
    M, N = A.shape
    row_sums = torch.zeros(M, device=A.device, dtype=A.dtype)
    grid = (M,)
    row_sum_kernel[grid](A, row_sums, M, N, BLOCK_N=BLOCK_N)
    return row_sums


def reference(A):
    return torch.sum(A, dim=1)


# --- Test 1: small fixed matrix ---
M, N, BLOCK_N = 4, 8, 8
A = torch.arange(M * N, dtype=torch.float32, device='cuda').reshape(M, N)
row_sums = run_kernel(A, BLOCK_N)
row_sums_ref = reference(A)

print("Matrix A:\n", A)
print("Kernel row sums:\n", row_sums)
print("Reference row sums:\n", row_sums_ref)
print("Match?", torch.allclose(row_sums, row_sums_ref))

# --- Test 2: random matrix ---
M, N, BLOCK_N = 7, 13, 16  # pad N=13 to the next power of two; mask the last 3 elements
A = torch.randn(M, N, device='cuda', dtype=torch.float32)
row_sums = run_kernel(A, BLOCK_N)
row_sums_ref = reference(A)

print("Random test, match?", torch.allclose(row_sums, row_sums_ref))

# --- Test 3: larger matrix ---
M, N, BLOCK_N = 64, 128, 128
A = torch.randn(M, N, device='cuda', dtype=torch.float32)
row_sums = run_kernel(A, BLOCK_N)
row_sums_ref = reference(A)

print("Large test, match?", torch.allclose(row_sums, row_sums_ref))



Matrix A:
 tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.],
        [16., 17., 18., 19., 20., 21., 22., 23.],
        [24., 25., 26., 27., 28., 29., 30., 31.]], device='cuda:0')
Kernel row sums:
 tensor([ 28.,  92., 156., 220.], device='cuda:0')
Reference row sums:
 tensor([ 28.,  92., 156., 220.], device='cuda:0')
Match? True
Random test, match? True
Large test, match? True


# Task 2: Column-wise Maximum

**Problem:**  

Implement a Triton kernel that computes **the maximum value in each column** of a matrix.  

- Input: `A` of shape `(M, N)`  
- Output: `col_max` of shape `(N,)`  
- Hint:  
  - Tile along the **M dimension** (rows)  
  - Use `tl.max` for reduction  
  - Mask out-of-bounds threads in the last tile

In [10]:
@triton.jit
def col_max_kernel(A_ptr: torch.Tensor,
                   col_max_ptr: torch.Tensor,
                   M: int, N: int,
                   BLOCK_M: tl.constexpr):
    col = tl.program_id(0)
    acc = tl.full((BLOCK_M,), float("-inf"), dtype=tl.float32)
    for start in range(0, M, BLOCK_M):
        rows = start + tl.arange(0, BLOCK_M)
        x = tl.load(A_ptr + rows * N + col, mask=rows < M, other=float("-inf"))
        acc = tl.maximum(acc, x)
    tl.store(col_max_ptr + col, tl.max(acc, axis=0))


# --- Helper functions ---
def run_col_max_kernel(A, BLOCK_M):
    M, N = A.shape
    col_max = torch.zeros(N, device=A.device, dtype=A.dtype)
    grid = (N,)
    col_max_kernel[grid](A, col_max, M, N, BLOCK_M=BLOCK_M)
    return col_max


def reference_col_max(A):
    return torch.max(A, dim=0).values


# --- Test 1: small fixed matrix ---
M, N, BLOCK_M = 6, 4, 8
A = torch.randn(M, N, device='cuda')
col_max = run_col_max_kernel(A, BLOCK_M)
col_max_ref = reference_col_max(A)

print("Matrix A:\n", A)
print("Kernel column max:\n", col_max)
print("Reference column max:\n", col_max_ref)
print("Match?", torch.allclose(col_max, col_max_ref))

# --- Test 2: random matrix not divisible by BLOCK_M ---
M, N, BLOCK_M = 7, 5, 8
A = torch.randn(M, N, device='cuda')
col_max = run_col_max_kernel(A, BLOCK_M)
col_max_ref = reference_col_max(A)

print("Random test, match?", torch.allclose(col_max, col_max_ref))

# --- Test 3: larger matrix ---
M, N, BLOCK_M = 64, 128, 64
A = torch.randn(M, N, device='cuda')
col_max = run_col_max_kernel(A, BLOCK_M)
col_max_ref = reference_col_max(A)

print("Large test, match?", torch.allclose(col_max, col_max_ref))

Matrix A:
 tensor([[-0.1756, -1.2057,  0.1576,  1.0054],
        [-0.3578, -1.4236,  0.3191,  0.8171],
        [ 0.0848,  0.5594, -1.1609, -1.2227],
        [ 1.8798, -0.2293,  0.5858, -0.4379],
        [-0.6032,  0.3599,  2.3306, -0.8751],
        [ 0.8264,  1.0376, -1.0708,  0.0113]], device='cuda:0')
Kernel column max:
 tensor([1.8798, 1.0376, 2.3306, 1.0054], device='cuda:0')
Reference column max:
 tensor([1.8798, 1.0376, 2.3306, 1.0054], device='cuda:0')
Match? True
Random test, match? True
Large test, match? True


# Task 3: GELU Activation

**Problem:**  

Implement the **GELU activation**:

$
\text{GELU}(x) = 0.5 \cdot x \cdot (1 + \tanh(\sqrt{2/\pi} \cdot (x + 0.044715 x^3)))
$

- Input: `X` of shape `(B, F)`  
- Output: `Y` of same shape  
- Hint: Tile along the feature dimension and apply elementwise operations using `tl.pow`, `tl.tanh`, and broadcasting

In [11]:
@triton.jit
def gelu_kernel(X_ptr: torch.Tensor,
                Y_ptr: torch.Tensor,
                B: int, F: int,
                BLOCK_F: tl.constexpr):
    row = tl.program_id(0)
    for start in range(0, F, BLOCK_F):
        cols = start + tl.arange(0, BLOCK_F)
        mask = cols < F
        x = tl.load(X_ptr + row * F + cols, mask=mask, other=0.0)
        inner = 0.7978845608028654 * (x + 0.044715 * x * x * x)  # sqrt(2 / pi) * (...)
        # tanh в tl нет, но 0.5 * (1 + tanh(z)) == sigmoid(2z)
        y = x * tl.sigmoid(2 * inner)
        tl.store(Y_ptr + row * F + cols, y, mask=mask)


# --- Helper functions ---
def run_gelu_kernel(X, BLOCK_F):
    B, F = X.shape
    Y = torch.zeros_like(X)
    grid = (B,)
    gelu_kernel[grid](X, Y, B, F, BLOCK_F=BLOCK_F)
    return Y


def reference_gelu(X):
    return torch.nn.functional.gelu(X, approximate="tanh")


# --- Test 1: small fixed matrix ---
B, F, BLOCK_F = 2, 8, 8
X = torch.randn(B, F, device='cuda')
Y = run_gelu_kernel(X, BLOCK_F)
Y_ref = reference_gelu(X)

print("X:\n", X)
print("Kernel GELU Y:\n", Y)
print("Reference GELU Y:\n", Y_ref)
print("Match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 2: random matrix not divisible by BLOCK_F ---
B, F, BLOCK_F = 3, 10, 16
X = torch.randn(B, F, device='cuda')
Y = run_gelu_kernel(X, BLOCK_F)
Y_ref = reference_gelu(X)

print("Random test, match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 3: larger matrix ---
B, F, BLOCK_F = 64, 128, 128
X = torch.randn(B, F, device='cuda')
Y = run_gelu_kernel(X, BLOCK_F)
Y_ref = reference_gelu(X)

print("Large test, match?", torch.allclose(Y, Y_ref, atol=1e-6))


X:
 tensor([[ 0.2469, -0.4572, -0.3041,  0.9629,  0.3817,  0.9743, -1.2228, -1.1424],
        [ 1.4149, -1.6460,  1.2901, -2.6712,  0.7013, -0.9191,  0.9212, -1.5479]],
       device='cuda:0')
Kernel GELU Y:
 tensor([[ 0.1475, -0.1480, -0.1157,  0.8012,  0.2476,  0.8135, -0.1356, -0.1449],
        [ 1.3035, -0.0823,  1.1628, -0.0096,  0.5318, -0.1647,  0.7567, -0.0944]],
       device='cuda:0')
Reference GELU Y:
 tensor([[ 0.1475, -0.1480, -0.1157,  0.8012,  0.2476,  0.8135, -0.1356, -0.1449],
        [ 1.3035, -0.0823,  1.1628, -0.0096,  0.5318, -0.1647,  0.7567, -0.0944]],
       device='cuda:0')
Match? True
Random test, match? True
Large test, match? True


## Task 4: Layer Normalization (Row-wise)

**Problem:**  

Implement **row-wise layer normalization** (without gamma/bamma).  

- Input: `X` of shape `(B, F)`  
- Output: `Y` of same shape  
- Formula:  

$
Y[i,j] = \frac{X[i,j] - \text{mean}_i}{\sqrt{\text{var}_i + \epsilon}}
$

**Hints:**  
- Compute **row mean and variance in tiles**  
- Use `tl.sum` and `tl.pow` for accumulation

In [12]:
import torch
import triton
import triton.language as tl

EPS = 1e-5


@triton.jit
def layer_norm_kernel(X_ptr: torch.Tensor,
                      Y_ptr: torch.Tensor,
                      B: int, F: int, eps: float,
                      BLOCK_F: tl.constexpr):
    row = tl.program_id(0)
    x_row = X_ptr + row * F
    y_row = Y_ptr + row * F

    acc = tl.zeros((BLOCK_F,), dtype=tl.float32)
    for start in range(0, F, BLOCK_F):
        cols = start + tl.arange(0, BLOCK_F)
        acc += tl.load(x_row + cols, mask=cols < F, other=0.0)
    mean = tl.sum(acc, axis=0) / F

    acc = tl.zeros((BLOCK_F,), dtype=tl.float32)
    for start in range(0, F, BLOCK_F):
        cols = start + tl.arange(0, BLOCK_F)
        x = tl.load(x_row + cols, mask=cols < F, other=0.0)
        diff = tl.where(cols < F, x - mean, 0.0)
        acc += diff * diff
    var = tl.sum(acc, axis=0) / F
    rstd = 1 / tl.sqrt(var + eps)

    for start in range(0, F, BLOCK_F):
        cols = start + tl.arange(0, BLOCK_F)
        mask = cols < F
        x = tl.load(x_row + cols, mask=mask, other=0.0)
        tl.store(y_row + cols, (x - mean) * rstd, mask=mask)


# --- Helper functions ---
def run_layer_norm_kernel(X, BLOCK_F):
    B, F = X.shape
    Y = torch.zeros_like(X)
    grid = (B,)
    layer_norm_kernel[grid](X, Y, B, F, EPS, BLOCK_F=BLOCK_F)
    return Y


def reference_layer_norm(X, eps=EPS):
    mean = torch.mean(X, dim=1, keepdim=True)
    var = torch.var(X, dim=1, unbiased=False, keepdim=True)
    return (X - mean) / torch.sqrt(var + eps)


# --- Test 1: small fixed matrix ---
B, F, BLOCK_F = 2, 8, 8
X = torch.randn(B, F, device='cuda')
Y = run_layer_norm_kernel(X, BLOCK_F)
Y_ref = reference_layer_norm(X)

print("X:\n", X)
print("Kernel LayerNorm Y:\n", Y)
print("Reference LayerNorm Y:\n", Y_ref)
print("Match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 2: random matrix not divisible by BLOCK_F ---
B, F, BLOCK_F = 3, 10, 16
X = torch.randn(B, F, device='cuda')
Y = run_layer_norm_kernel(X, BLOCK_F)
Y_ref = reference_layer_norm(X)

print("Random test, match?", torch.allclose(Y, Y_ref, atol=1e-6))

# --- Test 3: larger matrix ---
B, F, BLOCK_F = 64, 128, 128
X = torch.randn(B, F, device='cuda')
Y = run_layer_norm_kernel(X, BLOCK_F)
Y_ref = reference_layer_norm(X)

print("Large test, match?", torch.allclose(Y, Y_ref, atol=1e-6))


X:
 tensor([[ 3.2732e-01,  1.9919e+00, -1.2968e+00,  5.8696e-01,  1.5684e-01,
          8.1282e-01, -5.7367e-01, -9.7168e-01],
        [-7.5344e-01, -7.1407e-01, -1.0822e+00,  4.2698e-01, -8.4323e-02,
          5.0277e-02, -6.8962e-04,  1.5199e-01]], device='cuda:0')
Kernel LayerNorm Y:
 tensor([[ 0.1990,  1.8713, -1.4327,  0.4599,  0.0278,  0.6868, -0.7062, -1.1060],
        [-1.0155, -0.9359, -1.6794,  1.3687,  0.3360,  0.6079,  0.5049,  0.8133]],
       device='cuda:0')
Reference LayerNorm Y:
 tensor([[ 0.1990,  1.8713, -1.4327,  0.4599,  0.0278,  0.6868, -0.7062, -1.1060],
        [-1.0155, -0.9359, -1.6794,  1.3687,  0.3360,  0.6079,  0.5049,  0.8133]],
       device='cuda:0')
Match? True
Random test, match? True
Large test, match? True
